# Módulo 3.2: Mineração de Dados — Qualidade, Outliers, Tratamento de NAs e Regex Massivo

> **Resumo Didático Integrado** — Baseado no *Capítulo 2 (Mineração de Dados — Cadernos King County & Pesquisa Salarial)*.
> 
> Este caderno é **100% autocontido**: você aprenderá a diagnosticar e sanar falhas graves de qualidade em bases do mundo real, incluindo a **detecção matemática de Outliers (critério IQR de Tukey)**, **imputação de valores ausentes (`NA`)** e **harmonização textual com Expressões Regulares (Regex)**.

---

## 1. Visão Geral: As Três Dimensões da Qualidade de Dados

Quando coletamos dados reais, deparamo-nos com três grandes categorias de ruído:

1. **Valores Ausentes (*Missing Values* / `NA`)**: Registros incompletos decorrentes de falhas de preenchimento, quedas de conexão ou recusa do entrevistado.
   * *Remoção*: Descarte de observações quando a proporção for insignificante (< 3%) ou aleatória (MCAR).
   * *Imputação*: Substituição por medidas de tendência central (mediana para variáveis assimétricas, média para simétricas, moda para categóricas).
2. **Valores Discrepantes (*Outliers*)**: Medições que se afastam dramaticamente do padrão do restante do conjunto de dados. Podem ser **erros de digitação/sensor** (ex.: imóvel com 33 quartos quando o correto era 3) ou **eventos raros legítimos** (ex.: uma mansão de R$ 50 milhões em um bairro residencial comum).
3. **Inconsistência Textual e Categórica**: Dados preenchidos por humanos sem validação de formulário (ex.: *“US”*, *“USA”*, *“United States”*, *“U.S.A.”*, *“america”* para se referir ao mesmo país). Exigem sanitização com Expressões Regulares.

---

## 2. A Detecção Matemática de Outliers: O Critério de Tukey (IQR)

O método mais robusto e não paramétrico para identificar *outliers* é baseado no **Intervalo Interquartil (IQR - *Interquartile Range*)**:

$$IQR = Q_3 - Q_1$$

Onde $Q_1$ é o primeiro quartil (percentil 25%) e $Q_3$ é o terceiro quartil (percentil 75%). A partir do IQR, estabelecemos os limites de aceitação:

$$Limite\ Inferior = Q_1 - 1.5 	imes IQR$$
$$Limite\ Superior = Q_3 + 1.5 	imes IQR$$

* **Outlier Moderado**: Observação entre $1.5 	imes IQR$ e $3.0 	imes IQR$ além dos quartis.
* **Outlier Extremo**: Observação superior a $3.0 	imes IQR$ além dos quartis (quase sempre um erro grave de coleta).

---

## 3. Passo a Passo Prático de Higienização Avançada

Aplicaremos a metodologia didática:
* 🎯 **O que fazer**
* 💻 **Como fazer**
* 💡 **Por que fazer**

### Passo 1: Diagnóstico de Qualidade e Outliers na Base de Imóveis

🎯 **O que fazer**:
Carregar a base imobiliária (`datasets/kc_house_data_erro.csv`), inspecionar inconsistências flagrantes (como número absurdo de quartos ou preços negativos/nulos) e calcular os limites teóricos de Tukey.

💻 **Como fazer**:

In [ ]:
# Limpeza de ambiente
rm(list = ls())

# Configuração de dimensões gráficas no Jupyter
if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 11, repr.plot.height = 5)
}

caminho_kc <- if (file.exists("../datasets/kc_house_data_erro.csv")) {
  "../datasets/kc_house_data_erro.csv"
} else if (file.exists("datasets/kc_house_data_erro.csv")) {
  "datasets/kc_house_data_erro.csv"
} else {
  stop("Arquivo kc_house_data_erro.csv não encontrado!")
}

kc_dados <- read.csv(caminho_kc, stringsAsFactors = FALSE)

cat("Dimensões do dataset de imóveis: Linhas =", nrow(kc_dados), "| Colunas =", ncol(kc_dados), "\n")
cat("Resumo estatístico da variável 'bedrooms' (quartos):\n")
print(summary(kc_dados$bedrooms))

# Identificando o registro extremo:
quarto_extremo <- kc_dados[kc_dados$bedrooms > 10, c("id", "price", "bedrooms", "sqft_living")]
print("Registro com erro grotesco identificado:")
print(quarto_extremo)

# CÁLCULO FORMAL DOS LIMITES DE TUKEY (IQR) PARA A ÁREA CONSTRUÍDA (sqft_living)
q1 <- quantile(kc_dados$sqft_living, 0.25, na.rm = TRUE)
q3 <- quantile(kc_dados$sqft_living, 0.75, na.rm = TRUE)
iqr_area <- q3 - q1

lim_inferior <- q1 - 1.5 * iqr_area
lim_superior <- q3 + 1.5 * iqr_area

cat("\n--- Limites de Tukey para Área Construída ---\n")
cat("Q1 (25%):", q1, "| Q3 (75%):", q3, "| IQR:", iqr_area, "\n")
cat("Limite Inferior de Corte:", lim_inferior, "\n")
cat("Limite Superior de Corte (Outliers Acima):", lim_superior, "\n")

💡 **Por que fazer**:
O método IQR não é influenciado pelos próprios valores extremos (ao contrário da média e do desvio padrão, que são fortemente distorcidos por eles). Ele define uma fronteira estatisticamente justificada para isolar observações que comprometeriam a generalização de qualquer modelo de regressão linear.

### Passo 2: Saneamento e Correção de Inconsistências Imobiliárias

🎯 **O que fazer**:
Aplicar as correções estruturais na base de imóveis: corrigir a observação anômala de quartos, sanar banheiros inconsistentes e filtrar registros inválidos.

💻 **Como fazer**:

In [ ]:
kc_limpo <- kc_dados

# 1. Correção do imóvel com 33 quartos:
# O imóvel possui 1620 pés quadrados, o que evidencia que 33 foi um erro de digitação de 3 quartos
kc_limpo$bedrooms[kc_limpo$bedrooms == 33] <- 3

# 2. Filtragem de imóveis com preço ou quartos zerados (erros de registro)
kc_limpo <- subset(kc_limpo, price > 0 & bedrooms > 0)

# 3. Conversão e formatação da coluna de data (ex: '20141013T000000' -> '2014-10-13')
kc_limpo$data_venda <- as.Date(substr(kc_limpo$date, 1, 8), format = "%Y%m%d")

# 4. Verificação visual comparativa antes e depois no boxplot
par(mfrow = c(1, 2))
boxplot(kc_dados$bedrooms, main = "Quartos (Bruto com Erro)", col = "#e63946", ylab = "Número de Quartos")
boxplot(kc_limpo$bedrooms, main = "Quartos (Higienizado)", col = "#2a9d8f", ylab = "Número de Quartos")
par(mfrow = c(1, 1))

cat("Saneamento de imóveis concluído. Registros válidos:", nrow(kc_limpo), "\n")

💡 **Por que fazer**:
Erros de digitação óbvios (como digitar `33` no teclado numérico em vez de `3`) deformam a inclinação da reta de regressão. Corrigir o erro em vez de simplesmente descartar a linha preserva informações valiosas sobre o preço e localização do imóvel.

### Passo 3: Limpeza de Textos Livres e Harmonização com Regex (Pesquisa Salarial)

🎯 **O que fazer**:
Carregar a base de pesquisa salarial (`datasets/SalarySurvey2021.csv`), renomear as colunas extensas para termos analíticos concisos e harmonizar os nomes dos países utilizando Expressões Regulares (`grepl` e `gsub`).

💻 **Como fazer**:

In [ ]:
caminho_survey <- if (file.exists("../datasets/SalarySurvey2021.csv")) {
  "../datasets/SalarySurvey2021.csv"
} else if (file.exists("datasets/SalarySurvey2021.csv")) {
  "datasets/SalarySurvey2021.csv"
} else {
  stop("Arquivo SalarySurvey2021.csv não encontrado!")
}

survey_bruto <- read.csv(caminho_survey, stringsAsFactors = FALSE, check.names = FALSE)

# Seleção das colunas essenciais e renomeação simplificada
colunas_analise <- c(1, 2, 3, 4, 6, 8, 11)
survey <- survey_bruto[, colunas_analise]
names(survey) <- c("timestamp", "faixa_etaria", "setor", "cargo", "salario_anual", "moeda", "pais")

# Inspeção das variações de escrita no campo 'país'
cat("Exemplos de grafias brutas encontradas para países:\n")
print(head(unique(survey$pais), 15))

# HARMONIZAÇÃO MASSIVA COM EXPRESSÕES REGULARES:
# Padronizando variações dos Estados Unidos para a sigla 'USA'
pais_limpo <- trimws(tolower(survey$pais))

# Expressão regular identificando: usa, us, u.s., united states, america, united states of america
eh_usa <- grepl("^us$|^usa$|^u\\.s\\.|united states|america|u\\.s\\.a\\.", pais_limpo)
survey$pais_padronizado <- ifelse(eh_usa, "USA",
                           ifelse(grepl("united kingdom|england|scotland|uk|wales", pais_limpo), "UK",
                           ifelse(grepl("canada", pais_limpo), "Canada", "Outro")))

cat("\nDistribuição dos principais países após harmonização por Regex:\n")
print(table(survey$pais_padronizado))

💡 **Por que fazer**:
Campos de preenchimento livre em questionários geram centenas de grafias diferentes para o mesmo conceito (caixa alta, abreviações com ponto, espaços no início). Sem o uso de expressões regulares robustas com `ignore.case` e operadores lógicos de ancoragem (`^`, `$`), é impossível agrupar os dados por país para comparações de mercado.

### Passo 4: Saneamento Numérico de Valores com Formatação de Moeda

🎯 **O que fazer**:
Limpar strings numéricas que contêm vírgulas e pontos monetários (ex.: `"55,000"` ou `"$120,000.00"`), convertendo-as para valores numéricos contínuos em ponto flutuante.

💻 **Como fazer**:

In [ ]:
# Exemplo de conversão do campo de salário
# Passo 1: remover símbolos monetários ($) e separadores de milhar (vírgula)
salario_limpo_str <- gsub("[$, ]", "", survey$salario_anual)

# Passo 2: coerção segura para numérico
survey$salario_num <- as.numeric(salario_limpo_str)

# Passo 3: isolar a amostra padronizada (profissionais nos EUA recebendo em USD)
amostra_analitica <- subset(survey,
                            pais_padronizado == "USA" &
                            moeda == "USD" &
                            !is.na(salario_num) &
                            salario_num >= 15000 & salario_num <= 1000000)

cat("Amostra analítica conformada com sucesso!\n")
cat("Total de observações válidas e comparáveis:", nrow(amostra_analitica), "\n")
cat("Média salarial anual (USD):", round(mean(amostra_analitica$salario_num), 2), "\n")
cat("Mediana salarial anual (USD):", median(amostra_analitica$salario_num), "\n")

💡 **Por que fazer**:
Comparar salários brutos em moedas diferentes (libras esterlinas vs dólares vs ienes) ou calcular a média de uma coluna que contém texto misturado corrompe todo o estudo estatístico. Filtrar uma moeda homogênea e remover separadores garante que a base possa ser usada em regressões preditivas.

### Passo 5: Estratégias de Imputação de Valores Ausentes (`NA`)

🎯 **O que fazer**:
Demonstrar na prática como diagnosticar a proporção de `NA`s e aplicar imputação por mediana e por moda (para categorias).

💻 **Como fazer**:

In [ ]:
# Criando vetor com falhas para demonstração prática de imputação
valores_experimento <- c(2400, 2600, NA, 3100, 2900, NA, 15000, 2800) # Contém NA e um outlier (15000)

# 1. Por que usar a mediana e não a média em dados com assimetria ou outliers?
media_com_outlier <- mean(valores_experimento, na.rm = TRUE)
mediana_robusta <- median(valores_experimento, na.rm = TRUE)

cat("Média (distorcida pelo outlier):", round(media_com_outlier, 2), "\n")
cat("Mediana (robusta e imune a distorções):", mediana_robusta, "\n")

# Imputação vetorizada com ifelse:
valores_imputados <- ifelse(is.na(valores_experimento), mediana_robusta, valores_experimento)
cat("Vetor após imputação pela mediana:\n", valores_imputados, "\n")

# 2. Imputação de variáveis categóricas pela MODA (categoria mais frequente)
calc_moda <- function(vetor) {
  tabela <- table(vetor[!is.na(vetor)])
  return(names(tabela)[which.max(tabela)])
}

setor_amostra <- c("Tech", "Finance", "Tech", NA, "Saúde", "Tech", NA)
moda_setor <- calc_moda(setor_amostra)
setor_imputado <- ifelse(is.na(setor_amostra), moda_setor, setor_amostra)

cat("\nSetor mais frequente (Moda):", moda_setor, "\n")
cat("Setores após imputação pela moda:\n", setor_imputado, "\n")

💡 **Por que fazer**:
Excluir linhas toda vez que houver um `NA` pode eliminar mais da metade da base amostral, gerando viés de seleção. A imputação pela **mediana** é a melhor prática para variáveis quantitativas assimétricas, enquanto a **moda** é a técnica padrão para preencher atributos qualitativos sem inventar novas classes.

## 4. Conclusão e Próximos Passos
Neste Módulo 3.2, completamos o ciclo de Mineração e Tratamento de Dados:
* Identificação matemática e visual de **Outliers via critério IQR de Tukey**.
* Correção pontual de anomalias imobiliárias (`kc_house_data_erro.csv`).
* Tratamento de texto livre e harmonização de países com **Expressões Regulares**.
* Estratégias de **imputação de dados faltantes (`NA`)** pela mediana e pela moda.

Com os dados devidamente higienizados e confiáveis, estamos prontos para a fase de modelagem! No **Módulo 4 (`04_Analise_Exploratoria_e_Regressao_Linear_Simples.ipynb`)**, aprenderemos a construir modelos estatísticos preditivos com **Regressão Linear Simples**, interpretando coeficientes, erro padrão e métricas de desempenho.